In [ ]:
import numpy as np 
import pandas as pd 
import torch
from anglepy import ANGLE
from anglepy.metrics import mean_absolute_angular_deviation, circular_mean_directional_error, mean_circular_crps, accuracy, med_err
from anglepy.kernels import c2_wendland
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import math

# 1. German Wind Dataset - Prediction

In [ ]:
data = pd.read_csv("/kaggle/input/datasets/rajdeeppathak/wind-datasets/Germany_wind.csv")

def to_pi_range(theta):
    """Converts angles from [0, 2pi) to [-pi, pi]."""
    return (theta + np.pi) % (2 * np.pi) - np.pi

def to_2pi_range(theta):
    """Converts angles from [-pi, pi] to [0, 2pi)."""
    return theta % (2 * np.pi)

seed = 18
np.random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Split data into train and test
prec = 0.2 # Proportion of test data
data_shape = data.shape[0]
test_size = int(prec * data_shape)
test_ind = np.random.choice(data_shape, size=test_size, replace=False)
train_ind = np.setdiff1d(np.arange(data_shape), test_ind)
ind_un = test_ind
ind_obs = train_ind    


# Create necessary arrays for further use
combined_xs = data[['Longitude', 'Latitude']].values # All locations
X_train = torch.from_numpy(combined_xs[ind_obs,:]).float().to(device) # Train locations
X_test = torch.from_numpy(combined_xs[ind_un,:]).float().to(device) # Test locations

ys_all = data['dir'].values # All observed values
y_train = torch.from_numpy(ys_all[ind_obs]).float().to(device) # Train observed
y_test = torch.from_numpy(ys_all[ind_un]).float().to(device) # Test observed

d = combined_xs.shape[0] # Number of locations overall
du = y_test.shape[0] # Number of test locations

y_train = torch.from_numpy(to_2pi_range(y_train.cpu().numpy())).to(device)
y_test = torch.from_numpy(to_2pi_range(y_test.cpu().numpy())).to(device)
y_train = y_train.reshape(-1,1)

In [ ]:
# ANGLE-Geodesic

accuracies, mederrs, crpss, cmdes, maads, times = [], [], [], [], [], []

for _ in tqdm(range(50)):
    model = ANGLE(
        X_train, y_train, 
        num_layer=2, hidden_dim=128, noise_dim=64, noise_std=1,
    add_bn=False, resblock=False,
    print_every_nepoch=1000, print_times_per_epoch=1, 
    num_epochs=500, batch_size=None, 
    standardize=False, device=device,
    dist_method="geodesic", verbose=False, circular_indices=None,
    circular_projection="atan2", kernel_func=c2_wendland()
    )
    
    y_pred = model.predict(X_test)
    y_pred_ensemble = model.sample(X_test, sample_size=600)
    maad_mean = mean_absolute_angular_deviation(y_pred, y_test, return_degrees=True).item()
    cmde = circular_mean_directional_error(y_pred, y_test).item()
    crps = mean_circular_crps(y_pred_ensemble, y_test.reshape(-1,1), return_degrees=True)[0]
    acc = accuracy(y_pred, y_test, theta=torch.pi/6)
    median_err = med_err(y_pred, y_test)
    accuracies.append(round(acc,3))
    mederrs.append(round(median_err,3))
    crpss.append(round(crps, 3))
    maads.append(round(maad_mean, 3))
    cmdes.append(round(cmde,3))
    
print(f"{np.mean(accuracies):.2f}\pm{np.std(accuracies):.2f}")
print(f"{np.mean(mederrs):.2f}\pm{np.std(mederrs):.2f}")
print(f"{np.mean(crpss):.2f}\pm{np.std(crpss):.2f}")
print(f"{np.mean(maads):.2f}\pm{np.std(maads):.2f}")
print(f"{np.mean(cmdes):.2f}\pm{np.std(cmdes):.2f}")


In [ ]:
# ANGLE-Chordal

accuracies, mederrs, crpss, cmdes, maads, times = [], [], [], [], [], []

for _ in tqdm(range(50)):
    model = ANGLE(
        X_train, y_train, 
        num_layer=2, hidden_dim=128, noise_dim=64, noise_std=1,
    add_bn=False, resblock=False, gamma=1.5,
    print_every_nepoch=1000, print_times_per_epoch=1, 
    num_epochs=500, batch_size=None, 
    standardize=False, device=device,
    dist_method="chordal", verbose=False, circular_indices=None
    )
    
    y_pred = model.predict(X_test)
    y_pred_ensemble = model.sample(X_test, sample_size=600)
    maad_mean = mean_absolute_angular_deviation(y_pred, y_test, return_degrees=True).item()
    cmde = circular_mean_directional_error(y_pred, y_test).item()
    crps = mean_circular_crps(y_pred_ensemble, y_test.reshape(-1,1), return_degrees=True)[0]
    acc = accuracy(y_pred, y_test, theta=torch.pi/6)
    median_err = med_err(y_pred, y_test)
    accuracies.append(round(acc,3))
    mederrs.append(round(median_err,3))
    crpss.append(round(crps, 3))
    maads.append(round(maad_mean, 3))
    cmdes.append(round(cmde,3))
    
print(f"{np.mean(accuracies):.2f}\pm{np.std(accuracies):.2f}")
print(f"{np.mean(mederrs):.2f}\pm{np.std(mederrs):.2f}")
print(f"{np.mean(crpss):.2f}\pm{np.std(crpss):.2f}")
print(f"{np.mean(maads):.2f}\pm{np.std(maads):.2f}")
print(f"{np.mean(cmdes):.2f}\pm{np.std(cmdes):.2f}")

# 2. Indian Wind Dataset - Prediction

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/rajdeeppathak/wind-datasets/India_Wind_Data.csv')
random_state=18
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
X, y = df[['Latitude', 'Longitude']], df['Direction_Rad']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, shuffle=True, random_state=random_state)
X_train, X_test, y_train, y_test = torch.from_numpy(X_train.values).float().to(device), torch.from_numpy(X_test.values).float().to(device), torch.from_numpy(y_train.values).float().to(device), torch.from_numpy(y_test.values).float().to(device)
y_train = y_train.unsqueeze(-1)

In [ ]:
# ANGLE-Geodesic

accuracies, mederrs, crpss, cmdes, maads, times = [], [], [], [], [], []

for _ in tqdm(range(50)):
    model = ANGLE(
        X_train, y_train, 
        num_layer=3, hidden_dim=100, noise_dim=64, 
            add_bn=True, resblock=False, noise_std=1,
            print_every_nepoch=10000, print_times_per_epoch=0, 
            lr=0.05, num_epochs=500, batch_size=None, 
            standardize=False, device=device,
            dist_method='geodesic', verbose=False, circular_indices=None,
            kernel_func=c2_wendland(), circular_projection='atan2'
    )
    
    y_pred = model.predict(X_test)
    y_pred_ensemble = model.sample(X_test, sample_size=100)
    maad_mean = mean_absolute_angular_deviation(y_pred, y_test, return_degrees=True).item()
    cmde = circular_mean_directional_error(y_pred, y_test).item()
    crps = mean_circular_crps(y_pred_ensemble, y_test, return_degrees=True)[0]
    acc = accuracy(y_pred, y_test, theta=torch.pi/6)
    median_err = med_err(y_pred, y_test)
    accuracies.append(round(acc,3))
    mederrs.append(round(median_err,3))
    crpss.append(round(crps, 3))
    maads.append(round(maad_mean, 3))
    cmdes.append(round(cmde,3))
print(f"{np.mean(accuracies):.2f}\pm{np.std(accuracies):.2f}")
print(f"{np.mean(mederrs):.2f}\pm{np.std(mederrs):.2f}")
print(f"{np.mean(crpss):.2f}\pm{np.std(crpss):.2f}")
print(f"{np.mean(maads):.2f}\pm{np.std(maads):.2f}")
print(f"{np.mean(cmdes):.2f}\pm{np.std(cmdes):.2f}")

In [ ]:
# ANGLE-Chordal
accuracies, mederrs, crpss, cmdes, maads, times = [], [], [], [], [], []

for _ in tqdm(range(50)):
    model = ANGLE(
        X_train, y_train, 
        num_layer=3, hidden_dim=100, noise_dim=64, 
            add_bn=True, resblock=False, gamma=1, noise_std=1,
            print_every_nepoch=10000, print_times_per_epoch=0, 
            lr=0.05, num_epochs=800, batch_size=None, 
            standardize=False, device=device,
            dist_method='chordal', verbose=False, circular_indices=None
    )
    
    y_pred = model.predict(X_test)
    y_pred_ensemble = model.sample(X_test, sample_size=100)
    maad_mean = mean_absolute_angular_deviation(y_pred, y_test, return_degrees=True).item()
    cmde = circular_mean_directional_error(y_pred, y_test).item()
    crps = mean_circular_crps(y_pred_ensemble, y_test, return_degrees=True)[0]
    acc = accuracy(y_pred, y_test, theta=torch.pi/6)
    median_err = med_err(y_pred, y_test)
    accuracies.append(round(acc,3))
    mederrs.append(round(median_err,3))
    crpss.append(round(crps, 3))
    maads.append(round(maad_mean, 3))
    cmdes.append(round(cmde,3))

print(f"{np.mean(accuracies):.2f}\pm{np.std(accuracies):.2f}")
print(f"{np.mean(mederrs):.2f}\pm{np.std(mederrs):.2f}")
print(f"{np.mean(crpss):.2f}\pm{np.std(crpss):.2f}")
print(f"{np.mean(maads):.2f}\pm{np.std(maads):.2f}")
print(f"{np.mean(cmdes):.2f}\pm{np.std(cmdes):.2f}")


# 3. Indian Wind Dataset - Extrapolability

In [ ]:
csv_path = '/kaggle/input/datasets/rajdeeppathak/wind-datasets/India_Wind_Data.csv'

df = pd.read_csv(csv_path)

# Calculate the 85th percentiles for both columns
lat_q85 = df['Latitude'].quantile(0.85)
lon_q85 = df['Longitude'].quantile(0.85)

# Create a boolean mask for the training condition
# Both Latitude AND Longitude must be strictly lower than their 85th quantiles
train_mask = (df['Latitude'] < lat_q85) & (df['Longitude'] < lon_q85)

# Split the data into train and test sets
train_set = df[train_mask].copy()
test_set = df[~train_mask].copy() 

# Print the sizes of your splits to verify
print(f"Total records: {len(df)}")
print(f"Training set records: {len(train_set)} ({len(train_set)/len(df):.1%})")
print(f"Test set records: {len(test_set)} ({len(test_set)/len(df):.1%})")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train = train_set[['Latitude', 'Longitude']]
X_test = test_set[['Latitude', 'Longitude']]
y_train = train_set["Direction_Rad"]
y_test = test_set["Direction_Rad"]
X_train, X_test, y_train, y_test = torch.from_numpy(X_train.values).float().to(device), torch.from_numpy(X_test.values).float().to(device), torch.from_numpy(y_train.values).float().to(device), torch.from_numpy(y_test.values).float().to(device)
y_train = y_train.unsqueeze(-1)

In [ ]:
# ANGLE-Geodesic
accuracies, mederrs, crpss, cmdes, maads, times = [], [], [], [], [], []
kernel_func = c2_wendland(c=0.6119926142279882, tau=4)
params = {'num_layer': 4,
 'hidden_dim': 305,
 'noise_dim': 227,
 'noise_std': 1.033496979824373,
 'lr': 0.00021265021913055313,
 'add_bn': False,
 'resblock': False,
 'circular_projection':'sigmoid'}

for _ in tqdm(range(50)):
    model = ANGLE(
        X_train, y_train, 
        **params,
    print_every_nepoch=1000, print_times_per_epoch=1, 
    num_epochs=800, batch_size=None, 
    standardize=False, device=device,
     verbose=False, circular_indices=None, kernel_func=kernel_func
    )

    y_pred = model.predict(X_test)
    y_pred_ensemble = model.sample(X_test, sample_size=100)

    maad_mean = mean_absolute_angular_deviation(y_pred, y_test, return_degrees=True).item()
    cmde = circular_mean_directional_error(y_pred, y_test).item()
    crps = mean_circular_crps(y_pred_ensemble, y_test, return_degrees=True)[0]
    acc = accuracy(y_pred, y_test, theta=torch.pi/6)
    median_err = med_err(y_pred, y_test)
    accuracies.append(round(acc,3))
    mederrs.append(round(median_err,3))
    crpss.append(round(crps, 3))
    maads.append(round(maad_mean, 3))
    cmdes.append(round(cmde,3))

print(f"{np.mean(accuracies):.2f}\pm{np.std(accuracies):.2f}")
print(f"{np.mean(mederrs):.2f}\pm{np.std(mederrs):.2f}")
print(f"{np.mean(crpss):.2f}\pm{np.std(crpss):.2f}")
print(f"{np.mean(maads):.2f}\pm{np.std(maads):.2f}")
print(f"{np.mean(cmdes):.2f}\pm{np.std(cmdes):.2f}")


In [ ]:
# ANGLE-Chordal
accuracies, mederrs, crpss, cmdes, maads, times = [], [], [], [], [], []

for _ in tqdm(range(50)):
    model = ANGLE(
        X_train, y_train, 
        num_layer=2, hidden_dim=244, noise_dim=103,
    gamma=1.665114030259257,
    noise_std=3.6630288645664066,
    lr=0.0003751153035318678,
    add_bn=False,
    resblock=True,
    print_every_nepoch=1000, print_times_per_epoch=1, 
    num_epochs=800, batch_size=None, dist_method='chordal',
    standardize=False, device=device,
     verbose=False, circular_indices=None
    )

    y_pred = model.predict(X_test)
    y_pred_ensemble = model.sample(X_test, sample_size=100)

    maad_mean = mean_absolute_angular_deviation(y_pred, y_test, return_degrees=True).item()
    cmde = circular_mean_directional_error(y_pred, y_test).item()
    crps = mean_circular_crps(y_pred_ensemble, y_test, return_degrees=True)[0]
    acc = accuracy(y_pred, y_test, theta=torch.pi/6)
    median_err = med_err(y_pred, y_test)
    accuracies.append(round(acc,3))
    mederrs.append(round(median_err,3))
    crpss.append(round(crps, 3))
    maads.append(round(maad_mean, 3))
    cmdes.append(round(cmde,3))

print(f"{np.mean(accuracies):.2f}\pm{np.std(accuracies):.2f}")
print(f"{np.mean(mederrs):.2f}\pm{np.std(mederrs):.2f}")
print(f"{np.mean(crpss):.2f}\pm{np.std(crpss):.2f}")
print(f"{np.mean(maads):.2f}\pm{np.std(maads):.2f}")
print(f"{np.mean(cmdes):.2f}\pm{np.std(cmdes):.2f}")
